In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import MNIST
from torchvision import transforms

from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [3]:
transform = transforms.ToTensor()

train_dataset = MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [4]:
trainloader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

testloader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [5]:
class MyLinear(nn.Module):

    def __init__(self, in_features, out_features):

        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(out_features, in_features) * 0.01
        )

        self.bias = nn.Parameter(
            torch.zeros(out_features)
        )

    def forward(self, x):

        return x @ self.weight.T + self.bias

In [6]:
class MyNetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = MyLinear(784,256)

        self.relu1 = nn.ReLU()

        self.fc2 = MyLinear(256,128)

        self.relu2 = nn.ReLU()

        self.fc3 = MyLinear(128,10)

    def forward(self,x):

        x = x.view(x.size(0),-1)

        x = self.fc1(x)

        x = self.relu1(x)

        x = self.fc2(x)

        x = self.relu2(x)

        x = self.fc3(x)

        return x

In [7]:
model = MyNetwork().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [8]:
def train(model, trainloader, testloader, epochs):

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for images, labels in trainloader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in testloader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, pred = torch.max(outputs,1)

                total += labels.size(0)

                correct += (pred==labels).sum().item()

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss={running_loss/len(trainloader):.4f} | "
            f"Accuracy={100*correct/total:.2f}%"
        )

In [9]:
train(
    model,
    trainloader,
    testloader,
    epochs=5
)

Epoch 1/5 | Loss=0.3776 | Accuracy=95.10%
Epoch 2/5 | Loss=0.1389 | Accuracy=96.56%
Epoch 3/5 | Loss=0.0892 | Accuracy=97.38%
Epoch 4/5 | Loss=0.0655 | Accuracy=97.70%
Epoch 5/5 | Loss=0.0497 | Accuracy=97.47%
